# Full-Scale Bathymetry Workflow

This notebook demonstrates a **production-ready** bathymetry workflow using real GEBCO data and a realistic model grid. It uses `set_from_dataset` which auto-selects the appropriate pipeline.

## Before you start

You will need:
- A GEBCO netCDF file (global or regional, 15-arc-second or 30-arc-second)
- A model grid at the resolution you want to configure

Set the paths in the **Configuration** cell below.

## Configuration

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────

GEBCO_PATH = "/path/to/gebco.nc"   # <-- set this when you have the file

# Model domain
XSTART     = 260.0    # western longitude
LENX       = 30.0     # domain width (degrees)
YSTART     = 15.0     # southern latitude
LENY       = 20.0     # domain height (degrees)
RESOLUTION = 0.5      # model grid resolution (degrees)
GRID_NAME  = "gulf_of_mexico"

# Pipeline options
MASK_METHOD      = "ocean_frac"  # "ocean_frac" | "cartopy" | None
NX_SUB           = 5             # sub-points per cell in x (ocean_frac only)
NY_SUB           = 5             # sub-points per cell in y (ocean_frac only)
MASK_THRESHOLD   = 0.5
SMOOTH_SCL       = 2.0           # Cressman smoothing radius multiplier
CRESSMAN_EXP     = 2.0
MIN_DEPTH        = 5.0           # m
FILL_CHANNELS    = False

# Output
OUTPUT_DIR  = "/tmp/mom6_forge_output"    # where to write topog.nc, weights, etc.

# ──────────────────────────────────────────────────────────────────────────────

from pathlib import Path
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from mom6_forge.grid import Grid
from mom6_forge.topo import Topo
from mom6_forge._source_bathy import SourceBathy

plt.rcParams["figure.dpi"] = 120

## Step 1: Build the model grid

In [ ]:
grid = Grid(
    resolution=RESOLUTION,
    xstart=XSTART,
    lenx=LENX,
    ystart=YSTART,
    leny=LENY,
    name=GRID_NAME,
)

print(f"Model grid: {grid.nx} × {grid.ny} T-cells at {RESOLUTION}° resolution")
print(f"Lon: {float(grid.tlon.min()):.2f} – {float(grid.tlon.max()):.2f}")
print(f"Lat: {float(grid.tlat.min()):.2f} – {float(grid.tlat.max()):.2f}")
print(f"Total cells: {grid.nx * grid.ny:,}")

In [ ]:
fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, color="tan")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3)
ax.scatter(
    grid.tlon.values.ravel(), grid.tlat.values.ravel(),
    transform=ccrs.PlateCarree(),
    c="steelblue", s=2, label="T-cell centres"
)
ax.set_extent([XSTART-2, XSTART+LENX+2, YSTART-2, YSTART+LENY+2], crs=ccrs.PlateCarree())
ax.gridlines(draw_labels=True, linewidth=0.3, color="gray")
ax.set_title(f"{GRID_NAME} — {grid.nx}×{grid.ny} grid at {RESOLUTION}°")
plt.tight_layout()
plt.show()

## Step 2: Load and inspect the source bathymetry

In [ ]:
# Check GEBCO file exists before proceeding
gebco_path = Path(GEBCO_PATH)
if not gebco_path.exists():
    raise FileNotFoundError(
        f"GEBCO file not found: {gebco_path}\n"
        "Set GEBCO_PATH in the Configuration cell to the path of your GEBCO netCDF file."
    )

# Peek at the file structure
ds_peek = xr.open_dataset(gebco_path)
print("GEBCO file variables:", list(ds_peek.data_vars))
print("GEBCO file coords:", list(ds_peek.coords))
print("\nFirst variable shape:", ds_peek[list(ds_peek.data_vars)[0]].shape)
ds_peek.close()

In [ ]:
# Adjust coordinate and variable names to match your GEBCO file
# Standard GEBCO 2023: lon='lon', lat='lat', elevation='elevation'
LON_NAME       = "lon"
LAT_NAME       = "lat"
ELEVATION_NAME = "elevation"

topo = Topo(grid, min_depth=MIN_DEPTH, version_control_dir=output_dir)
topo.set_flat(1000)

src = SourceBathy(
    gebco_path,
    lon_name=LON_NAME,
    lat_name=LAT_NAME,
    elevation_name=ELEVATION_NAME,
)

# Slice to domain — loads and clips the relevant region
print("Slicing GEBCO to domain (may take a moment for large files)...")
src = src.slice_to_domain(topo)

print(f"Sliced source shape: {src.depth.shape}  ({src.depth.shape[0] * src.depth.shape[1]:,} pixels)")
print(f"Source lon: {float(src.lon.min()):.3f} – {float(src.lon.max()):.3f}")
print(f"Source lat: {float(src.lat.min()):.3f} – {float(src.lat.max()):.3f}")
print(f"Depth range: {float(src.depth.min()):.1f} – {float(src.depth.max()):.1f} m")

In [ ]:
fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(111, projection=ccrs.PlateCarree())
c = ax.pcolormesh(
    src.lon, src.lat, src.depth,
    transform=ccrs.PlateCarree(),
    cmap="Blues", vmin=0, vmax=6000, shading="auto"
)
plt.colorbar(c, ax=ax, label="Depth (m, +down)")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.scatter(
    grid.tlon.values.ravel(), grid.tlat.values.ravel(),
    transform=ccrs.PlateCarree(),
    c="red", s=1, alpha=0.5, label="Model T-cells"
)
ax.set_extent([XSTART-1, XSTART+LENX+1, YSTART-1, YSTART+LENY+1], crs=ccrs.PlateCarree())
ax.gridlines(draw_labels=True, linewidth=0.3)
ax.set_title("GEBCO source bathymetry (sliced to domain)")
plt.tight_layout()
plt.show()

## Step 3: Diagnose resolution — which pipeline?

`diagnose_resolution` computes the ratio of average source pixel spacing to model cell spacing. If ≥ 12×, the high-res (Cressman) pipeline is recommended.

In [ ]:
use_high_res = topo.diagnose_resolution(src)
pipeline = "high_res_regrid (Cressman)" if use_high_res else "direct_xesmf_regrid"
print(f"diagnose_resolution → {pipeline}")

## Step 4: Generate the ocean mask

We generate the mask before `set_from_dataset` so we can inspect it. You can pass it directly via the `mask=` parameter to skip regeneration.

In [ ]:
if MASK_METHOD == "ocean_frac":
    print(f"Generating ocean_frac mask ({NX_SUB}×{NY_SUB} sub-points)...")
    mask = topo.generate_mask_ocean_frac(
        src,
        nx_sub=NX_SUB,
        ny_sub=NY_SUB,
        mask_threshold=MASK_THRESHOLD,
    )
elif MASK_METHOD == "cartopy":
    print("Generating cartopy mask...")
    mask = topo.generate_mask_cartopy(resolution="10m")
else:
    mask = None
    print("No pre-computed mask — will derive from regridded depth sign")

if mask is not None:
    n_ocean = int(mask.sum())
    n_total = mask.size
    print(f"\nMask: {n_ocean}/{n_total} ocean cells ({100*n_ocean/n_total:.1f}%)")

In [ ]:
if mask is not None:
    fig = plt.figure(figsize=(10, 5))
    ax = fig.add_subplot(111, projection=ccrs.PlateCarree())
    c = ax.pcolormesh(
        grid.qlon.values, grid.qlat.values, mask.values,
        transform=ccrs.PlateCarree(),
        cmap="RdBu", vmin=0, vmax=1, shading="flat"
    )
    plt.colorbar(c, ax=ax, label="0=land, 1=ocean")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.set_extent([XSTART-1, XSTART+LENX+1, YSTART-1, YSTART+LENY+1], crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=True, linewidth=0.3)
    ax.set_title(f"Ocean mask — method: {MASK_METHOD}")
    plt.tight_layout()
    plt.show()

## Step 5: Run the pipeline

We call the individual pipeline methods directly so we can inspect intermediate results. You could equivalently call `topo.set_from_dataset(GEBCO_PATH, mask=mask, mask_method=MASK_METHOD, ...)` to do this in one shot.

In [ ]:
weights_path = output_dir / "cressman_weights.nc"

if use_high_res:
    print("Running high_res_regrid (Cressman)...")
    topo.high_res_regrid(
        src,
        mask=mask,
        mask_method=MASK_METHOD if mask is None else None,
        nx_sub=NX_SUB,
        ny_sub=NY_SUB,
        mask_threshold=MASK_THRESHOLD,
        smooth_scl=SMOOTH_SCL,
        cressman_exp=CRESSMAN_EXP,
        hmin=MIN_DEPTH,
        weights_path=weights_path,
        fill_channels=FILL_CHANNELS,
    )
else:
    print("Running direct_xesmf_regrid...")
    topo.direct_xesmf_regrid(
        src,
        regridding_method="bilinear",
        mask=mask,
        mask_method=MASK_METHOD if mask is None else None,
        fill_channels=FILL_CHANNELS,
    )

print("Pipeline complete.")

## Step 6: Inspect and validate the result

In [ ]:
depth = topo.depth.values
ocean_cells = depth > 0

print("=== Depth statistics ===")
print(f"Grid size:         {grid.ny} × {grid.nx} = {grid.ny*grid.nx:,} cells")
print(f"Ocean cells:       {ocean_cells.sum():,} ({100*ocean_cells.mean():.1f}%)")
print(f"Land cells:        {(~ocean_cells).sum():,}")
print(f"Min ocean depth:   {depth[ocean_cells].min():.2f} m")
print(f"Max ocean depth:   {depth[ocean_cells].max():.2f} m")
print(f"Mean ocean depth:  {depth[ocean_cells].mean():.1f} m")
print(f"Median ocean depth:{np.median(depth[ocean_cells]):.1f} m")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5),
                          subplot_kw={"projection": ccrs.PlateCarree()})

# Depth field
ax = axes[0]
depth_plot = np.where(ocean_cells, depth, np.nan)
c = ax.pcolormesh(
    grid.qlon.values, grid.qlat.values, depth_plot,
    transform=ccrs.PlateCarree(),
    cmap="Blues_r", vmin=0, vmax=depth[ocean_cells].max(), shading="flat"
)
plt.colorbar(c, ax=ax, label="Depth (m)", shrink=0.8)
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.LAND, color="tan", zorder=3)
ax.set_extent([XSTART-1, XSTART+LENX+1, YSTART-1, YSTART+LENY+1], crs=ccrs.PlateCarree())
ax.gridlines(draw_labels=True, linewidth=0.3)
ax.set_title("Final ocean depth")

# Depth histogram
ax = axes[1]
ax.hist(depth[ocean_cells], bins=50, color="steelblue", edgecolor="none")
ax.axvline(MIN_DEPTH, color="red", linestyle="--", label=f"min_depth = {MIN_DEPTH} m")
ax.set_xlabel("Depth (m)")
ax.set_ylabel("Number of cells")
ax.set_title("Ocean depth distribution")
ax.legend()

plt.suptitle(f"{GRID_NAME} — {pipeline if 'pipeline' in dir() else 'pipeline'}", fontsize=12)
plt.tight_layout()
plt.show()

## Step 7: Write the topography file

In [ ]:
topog_path = output_dir / "topog.nc"
topo.write_topo(topog_path)
print(f"Written: {topog_path}")

# Inspect what was written
ds_out = xr.open_dataset(topog_path)
print("\nOutput variables:", list(ds_out.data_vars))
ds_out

## Step 8 (optional): Compare mask methods

If you want to compare `ocean_frac` vs `cartopy` masks on the same domain, run the cells below.

In [ ]:
# Generate cartopy mask for comparison
mask_cartopy = topo.generate_mask_cartopy(resolution="10m")

if mask is not None:
    disagree = (mask.values != mask_cartopy.values)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4),
                              subplot_kw={"projection": ccrs.PlateCarree()})

    for ax, data, title in [
        (axes[0], mask.values,         f"{MASK_METHOD} mask"),
        (axes[1], mask_cartopy.values, "cartopy mask (10m)"),
        (axes[2], disagree.astype(int), "Disagreement"),
    ]:
        c = ax.pcolormesh(
            grid.qlon.values, grid.qlat.values, data,
            transform=ccrs.PlateCarree(),
            cmap="RdBu", vmin=0, vmax=1, shading="flat"
        )
        plt.colorbar(c, ax=ax, shrink=0.8)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.set_extent([XSTART-1, XSTART+LENX+1, YSTART-1, YSTART+LENY+1],
                      crs=ccrs.PlateCarree())
        ax.gridlines(linewidth=0.3)
        ax.set_title(title)

    axes[2].set_title(f"Disagreement ({disagree.sum()} cells = {100*disagree.mean():.1f}%)")

    plt.suptitle("Mask method comparison", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No pre-computed mask to compare (mask=None was used).")

## Step 9 (optional): Topographic drag (`h²`)

If the `ocean_frac` mask method was used, per-cell depth statistics are available and `write_topo` will include the `h²` (topographic roughness variance) field used by MOM6's topographic drag parameterisation.

In [ ]:
if "h2" in ds_out:
    fig = plt.figure(figsize=(9, 5))
    ax = fig.add_subplot(111, projection=ccrs.PlateCarree())
    h2 = ds_out["h2"].values
    c = ax.pcolormesh(
        grid.qlon.values, grid.qlat.values, h2,
        transform=ccrs.PlateCarree(),
        cmap="hot_r", vmin=0, shading="flat"
    )
    plt.colorbar(c, ax=ax, label="h² (m²)", shrink=0.8)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.LAND, color="tan", zorder=3)
    ax.set_extent([XSTART-1, XSTART+LENX+1, YSTART-1, YSTART+LENY+1], crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=True, linewidth=0.3)
    ax.set_title("Topographic roughness h² = D2_mean - D_mean²")
    plt.tight_layout()
    plt.show()
    print(f"h² range: {h2.min():.1f} – {h2.max():.1f} m²")
else:
    print("h2 not in output — run with mask_method='ocean_frac' to get topographic drag statistics.")

## Summary of outputs

After running this notebook you have:

| File | Contents |
|---|---|
| `topog.nc` | `depth` (m), `h2` (m², if ocean_frac mask) — ready for MOM6 |
| `cressman_weights.nc` | ESMF-compatible sparse weight matrix — reusable |

The `topog.nc` file can be passed directly to MOM6 via `TOPO_FILE` in `MOM_input`.

---

## One-shot equivalent

Once you are satisfied with the configuration, the entire workflow above is equivalent to:

```python
from mom6_forge.grid import Grid
from mom6_forge.topo import Topo

grid = Grid(resolution=RESOLUTION, xstart=XSTART, lenx=LENX,
            ystart=YSTART, leny=LENY, name=GRID_NAME)
topo = Topo(grid, min_depth=MIN_DEPTH, version_control_dir=output_dir)
topo.set_flat(1000)

topo.set_from_dataset(
    GEBCO_PATH,
    mask_method=MASK_METHOD,
    nx_sub=NX_SUB,
    ny_sub=NY_SUB,
    mask_threshold=MASK_THRESHOLD,
    smooth_scl=SMOOTH_SCL,
    cressman_exp=CRESSMAN_EXP,
    hmin=MIN_DEPTH,
    weights_path=output_dir / "cressman_weights.nc",
    fill_channels=FILL_CHANNELS,
)

topo.write_topo(output_dir / "topog.nc")
```